# S3ANet Automated Experiments
This notebook runs the S3ANet model on all datasets (PaviaU, Salinas, Houston, IndianP), stores the results in proper subfolders, and generates a final Excel report.


## 1. Mount Google Drive and Setup Environment
Assuming your dataset `.mat` files are stored in your Google Drive under a folder named `SACNEet_Data`.
Please modify the `drive_datasets_path` below if they are stored in a different folder.


In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Ensure the S3ANet project is your current working directory
# Note: In Colab, you might need to clone your repo first if not already in it.
import os
if not os.path.exists('/content/S3ANET'):
    !git clone https://github.com/SRUJANPATEL3669/S3ANET.git
%cd /content/S3ANET

# Create Data folder
os.makedirs('./Data', exist_ok=True)

# Path to the datasets in your Google Drive (Modify if needed)
drive_datasets_path = '/content/drive/MyDrive/SACNet_Data'

if os.path.exists(drive_datasets_path):
    print("Copying datasets from Google Drive...")
    for file in os.listdir(drive_datasets_path):
        if file.endswith('.mat'):
            shutil.copy(os.path.join(drive_datasets_path, file), './Data/')
    print("Datasets copied successfully!")
else:
    print(f"Warning: The path {drive_datasets_path} does not exist in your Google Drive.")
    print("Please make sure the datasets are present in that folder.")


## 2. Generate Data Samples (`.npy` files)
This step runs `GenSample.py` for datasets 1 to 4 to prepare the training/testing arrays.


In [ ]:
# Run GenSample.py for datasets 1 to 4
for data_id in range(1, 5):
    print(f"\nGenerating samples for Data ID {data_id}...")
    !python GenSample.py --dataID {data_id}
print("\nData generation complete.")


## 3. Run FGSM Attack Experiments and Collect Results
This cell executes `Attack_FGSM_S3ANet.py` for all datasets and captures all metrics:
- **OA, Kappa, AA** — classification accuracy
- **SAM** — mean spectral angle of perturbation (physical-distance metric, degrees)
- **SID** — spectral information divergence (material-identity metric)
- **Physical-consistency rate** — fraction of adversarial pixels passing unmixing round-trip (θ=5°)
- **ASR** — attack success rate (fraction of correctly-classified pixels flipped)

Results are exported to `Attack_Results.xlsx`.


In [ ]:
# Run GenAdvExample.py (Adversarial Example Generation)
# Results will be saved directly to Google Drive
drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
import os
os.makedirs(drive_results_path, exist_ok=True)

for data_id in range(1, 5):
    print(f'\nGenerating adversarial examples for Data ID {data_id}...')
    !python GenAdvExample.py --dataID {data_id} --save_path_prefix {drive_results_path}
print('\nAdversarial example generation complete.')


In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
all_results = []

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

for data_id, dataset_name in datasets.items():
    print(f"\n================ Running Attack Experiment for {dataset_name} ================")
    
    save_prefix = f"/content/drive/MyDrive/S3ANet_Results/Attack_Results/{dataset_name}/"
    os.makedirs(save_prefix, exist_ok=True)
    
    # Run the script and capture output
    command = f"python Attack_FGSM_S3ANet.py --dataID {data_id} --save_path_prefix {save_prefix}"
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    output_lines = []
    for line in process.stdout:
        print(line, end='')  # Print out the logs in real-time
        output_lines.append(line.strip())
    process.wait()
    
    # ── Parse standard classification metrics ──────────────────────────────
    oa, kappa, aa, train_time, test_time, runtime = None, None, None, None, None, None
    producer_a = ""
    
    # ── Parse new spectral / physical attack metrics ───────────────────────
    sam_val, sid_val, phys_rate, asr_val = None, None, None, None
    
    for line in output_lines:
        # OA / Kappa
        match = re.search(r'OA=([\d\.]+),Kappa=([\d\.]+)', line)
        if match:
            oa    = float(match.group(1))
            kappa = float(match.group(2))
        
        # Producer accuracies
        if "producerA:" in line:
            producer_a = line.split("producerA:")[1].strip()
        
        # AA
        match = re.search(r'AA=([\d\.]+)', line)
        if match:
            aa = float(match.group(1))
        
        # Timing
        match = re.search(r'Train_time: ([\d\.]+), Test_time: ([\d\.]+), Runtime: ([\d\.]+)', line)
        if match:
            train_time = float(match.group(1))
            test_time  = float(match.group(2))
            runtime    = float(match.group(3))
        
        # SAM  (mean spectral angle, deg)  : 2.3147
        match = re.search(r'SAM\s*\(mean spectral angle.*?\)\s*:\s*([\d\.]+)', line)
        if match:
            sam_val = float(match.group(1))
        
        # SID  (spectral info divergence)  : 0.000341
        match = re.search(r'SID\s*\(spectral info.*?\)\s*:\s*([\d\.]+)', line)
        if match:
            sid_val = float(match.group(1))
        
        # Physical-consistency rate (θ=5°) : 0.9821  (98.21%)
        match = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if match:
            phys_rate = float(match.group(1))
        
        # ASR  (attack success rate)        : 0.6438  (64.38%)
        match = re.search(r'ASR\s*\(attack success.*?\)\s*:\s*([\d\.]+)', line)
        if match:
            asr_val = float(match.group(1))
    
    # Append to results
    all_results.append({
        'Dataset':                    dataset_name,
        'OA (%)':                     oa,
        'Kappa (%)':                  kappa,
        'AA (%)':                     aa,
        'Producer A':                 producer_a,
        'Train Time (s)':             train_time,
        'Test Time (s)':              test_time,
        'Total Runtime (s)':          runtime,
        'SAM (deg)':                  sam_val,
        'SID':                        sid_val,
        'Physical Consistency Rate':  phys_rate,
        'ASR':                        asr_val,
    })

print("\nAll attack experiments finished. Generating Excel report...")

# Create DataFrame and save to Excel
df_attack = pd.DataFrame(all_results)
excel_path = f'{drive_results_path}Attack_Results.xlsx'
df_attack.to_excel(excel_path, index=False)
print(f"Attack results saved to {excel_path}!")
df_attack  # Display the dataframe in Colab


## 4. Clean Test — S3ANet on Raw (Unperturbed) Data
This section trains **S3ANet** from scratch on each dataset and evaluates it on the **clean (unperturbed) test set** using `Test_Clean_S3ANet.py`.

Metrics reported:
- **OA, Kappa, AA** — classification accuracy
- **SAM = 0** — reference (no perturbation)
- **SID = 0** — reference (no perturbation)
- **Physical-consistency rate** — baseline unmixing round-trip quality of the clean image (θ=5°)
- **ASR = 0** — reference (no attack applied)

Results are exported to `Clean_Results.xlsx`.


In [ ]:
import subprocess
import pandas as pd
import re
import os

datasets = {1: 'PaviaU', 2: 'Salinas', 3: 'Houston', 4: 'IndianP'}
clean_results = []

drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
os.makedirs(drive_results_path, exist_ok=True)

for data_id, dataset_name in datasets.items():
    print(f'\n================ Clean Test for {dataset_name} ================')

    save_prefix = f'{drive_results_path}Clean_Results/{dataset_name}/'
    os.makedirs(save_prefix, exist_ok=True)

    # Run Test_Clean_S3ANet.py and capture output
    command = f'python Test_Clean_S3ANet.py --dataID {data_id} --save_path_prefix {save_prefix}'
    process = subprocess.Popen(
        command, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    output_lines = []
    for line in process.stdout:
        print(line, end='')          # real-time logging
        output_lines.append(line.strip())
    process.wait()

    # ── Parse standard classification metrics ──────────────────────────────
    # Test_Clean_S3ANet.py prints:
    #   OA      : 92.345 %
    #   Kappa   : 90.123 %
    #   AA      : 88.456 %
    #   Class  0: 95.000 %  ...
    oa, kappa, aa = None, None, None
    producer_a_list = []

    # ── Parse new spectral / physical metrics (clean baseline) ─────────────
    # Test_Clean_S3ANet.py prints:
    #   SAM  (mean spectral angle, deg)  : 0.0000  [0 = no perturbation]
    #   SID  (spectral info divergence)  : 0.000000  [0 = no perturbation]
    #   Physical-consistency rate (θ=5°) : 0.9900  (99.00%)
    #   ASR  (attack success rate)        : 0.0000  [0 = no attack]
    sam_val, sid_val, phys_rate, asr_val = None, None, None, None

    for line in output_lines:
        # OA
        m = re.search(r'OA\s*:\s*([\d\.]+)', line)
        if m:
            oa = float(m.group(1))

        # Kappa
        m = re.search(r'Kappa\s*:\s*([\d\.]+)', line)
        if m:
            kappa = float(m.group(1))

        # AA
        m = re.search(r'AA\s*:\s*([\d\.]+)', line)
        if m:
            aa = float(m.group(1))

        # Per-class Producer Accuracy
        m = re.search(r'Class\s*(\d+)\s*:\s*([\d\.]+)', line)
        if m:
            producer_a_list.append(f'C{m.group(1)}={m.group(2)}%')

        # SAM
        m = re.search(r'SAM\s*\(mean spectral angle.*?\)\s*:\s*([\d\.]+)', line)
        if m:
            sam_val = float(m.group(1))

        # SID
        m = re.search(r'SID\s*\(spectral info.*?\)\s*:\s*([\d\.]+)', line)
        if m:
            sid_val = float(m.group(1))

        # Physical-consistency rate
        m = re.search(r'Physical-consistency rate.*?:\s*([\d\.]+)', line)
        if m:
            phys_rate = float(m.group(1))

        # ASR
        m = re.search(r'ASR\s*\(attack success.*?\)\s*:\s*([\d\.]+)', line)
        if m:
            asr_val = float(m.group(1))

    clean_results.append({
        'Dataset':                    dataset_name,
        'OA (%)':                     oa,
        'Kappa (%)':                  kappa,
        'AA (%)':                     aa,
        'Producer A':                 ', '.join(producer_a_list),
        'SAM (deg)':                  sam_val,
        'SID':                        sid_val,
        'Physical Consistency Rate':  phys_rate,
        'ASR':                        asr_val,
    })

print('\nAll clean-test experiments finished. Generating Excel report...')

df_clean = pd.DataFrame(clean_results)
clean_excel_path = f'{drive_results_path}Clean_Results.xlsx'
df_clean.to_excel(clean_excel_path, index=False)
print(f'Clean results saved to {clean_excel_path}!')
df_clean


## 5. Side-by-Side Comparison: Attack vs Clean
Merge the attack and clean result tables into one comparison view for easy analysis.


In [ ]:
import pandas as pd

# Load both Excel sheets (or reuse in-memory DataFrames if still in session)
try:
    df_attack
    df_clean
except NameError:
    drive_results_path = '/content/drive/MyDrive/S3ANet_Results/'
    df_attack = pd.read_excel(f'{drive_results_path}Attack_Results.xlsx')
    df_clean  = pd.read_excel(f'{drive_results_path}Clean_Results.xlsx')

metric_cols = ['Dataset', 'OA (%)', 'Kappa (%)', 'AA (%)',
               'SAM (deg)', 'SID', 'Physical Consistency Rate', 'ASR']

df_cmp = df_clean[metric_cols].merge(
    df_attack[metric_cols],
    on='Dataset', suffixes=(' (Clean)', ' (Attack)')
)

# Save comparison
cmp_path = f'{drive_results_path}Comparison_Clean_vs_Attack.xlsx'
df_cmp.to_excel(cmp_path, index=False)
print(f'Comparison table saved to {cmp_path}!')
df_cmp
